# <span style='color: darkblue'>Building a Spam Filter with Naive Bayes</span>

## <span style='color: blue'>Introduction</span>

The aim of this project is to build a spam filter for SMS messages using the multinomial Naive Bayes algorithm that will classify messages as spam correctly at least 80% of the time.

A dataset compiled by Tiago A Almeida and José Gómez Hidalgo will be used to train the algorithm.

The dataset can be downloaded from [the UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) with details of the data collection process available [here](http://www.dt.fee.unicamp.br/~tiago/smsspamcollection/#composition).

Please note that due to the nature of spam messages, the dataset contains content that some readers may find offensive.

## <span style='color: blue'>Exploring the Dataset</span>

*Import required packages and load the dataset in to a DataFrame.*

*Note that the file does not have a file extension, is tab separated and does not contain a header row.*

In [1]:
import pandas as pd
import re

In [2]:
messages = pd.read_csv('SMSSpamCollection',
                       sep='\t',
                       header=None,
                       names=['Label', 'SMS'])

*Show how may rows and columns are in the dataset and the percentages of spam and ham (non-spam) messages.*

In [3]:
print('\nThe dataset contains {:,} rows and {} columns.\n'.format(messages.shape[0], messages.shape[1]))


The dataset contains 5,572 rows and 2 columns.



In [4]:
def calc_pcts(df):
    spam_proportions = df['Label'].value_counts(normalize=True)
    pct_ham = spam_proportions['ham'] * 100
    pct_spam = spam_proportions['spam'] * 100
    print('\n{:.1f}% of the messages in the dataset have been classified as spam.'.format(pct_spam))
    print('{:.1f}% of the messages have been classified as ham (non-spam)\n'.format(pct_ham))

In [5]:
calc_pcts(messages)


13.4% of the messages in the dataset have been classified as spam.
86.6% of the messages have been classified as ham (non-spam)



## <span style='color: blue'>Training and Test Set</span>

In order to test the spam filter, the dataset shall be split in to training and test datasets. The data will be split 80% for training and 20% for testing. The testing data shall be treated as if they were 'new' messages.

*Randomise the entire dataset making ensuring that the results are reproducible.*

In [6]:
messages = (messages.sample(frac=1,
                           random_state=1)
            .reset_index(drop=True))

*Calculate the cutover index number.*

In [7]:
cutover_index = round(len(messages) * 0.8)

*Create the two new datasets.*

In [8]:
training = messages[:cutover_index].reset_index(drop=True)
testing = messages[cutover_index:].reset_index(drop=True)

*Recalculate the percentages of spam and ham for each dataset.*

In [9]:
calc_pcts(training)


13.5% of the messages in the dataset have been classified as spam.
86.5% of the messages have been classified as ham (non-spam)



In [10]:
calc_pcts(testing)


13.2% of the messages in the dataset have been classified as spam.
86.8% of the messages have been classified as ham (non-spam)



The above show that the proportion of spam to non-spam messages in the split datasets is similar to that of the main dataset.

## <span style='color: blue'>Letter Case and Puntuation</span>

*Remove punctuation and convert the *`SMS`* column to lower case to allow the easy extraction of words for analysis.*

In [11]:
training['SMS'] = training['SMS'].str.replace('\W', ' ').str.lower()

## <span style='color: blue'>Creating the Vocabulary</span>

*Create a list of unique words generated from the SMS column.*

In [12]:
vocabulary = set()

In [13]:
training['SMS'] = training['SMS'].str.split()
for message in training['SMS']:
    for word in message:
        vocabulary.add(word)

In [14]:
vocabulary = list(vocabulary)
print('\nThere are {:,} unique words in the vocabulary list.\n'.format(len(vocabulary)))


There are 7,783 unique words in the vocabulary list.



## <span style='color: blue'>The Final Training Set</span>

*Create a dictionary of unique words from vocabulary that will contain a count of their appearances in the SMS column.*

In [15]:
word_counts = {word: [0] * len(training['SMS']) for word in vocabulary}
for i, msg in enumerate(training['SMS']):
    for word in msg:
        word_counts[word][i] += 1

*Convert the dictionary to a Pandas DataFrame.*

In [16]:
word_counts = pd.DataFrame(word_counts)

*Concatenate the new DataFrame with the training DataFrame.*

In [17]:
training = pd.concat([training, word_counts], axis=1)

## <span style='color: blue'>Calculating Constants First</span>

*Calculate probabilities for spam and ham messages.*

In [18]:
def probs(series, criteria):
    total = len(series)
    subtotal = len(series[series['Label'] == criteria])
    return subtotal / total

In [19]:
p_spam = probs(training, 'spam')
p_ham = probs(training, 'ham')
print('\nThe probability of spam in the training dataset is {:.2f}.'.format(p_spam))
print('The probability of ham in the training dataset is {:.2f}.\n'.format(p_ham))


The probability of spam in the training dataset is 0.13.
The probability of ham in the training dataset is 0.87.



*Calculate the number of words in spam messages, ham messages and the total number of unique words in vocabulary.*

In [20]:
def word_count(criteria):
    count = 0
    for msg in training[training['Label'] == criteria]['SMS']:
        count += len(msg)
    return count

In [21]:
n_spam = word_count('spam')
n_ham = word_count('ham')
n_vocabulary = len(vocabulary)
alpha = 1 # Laplace smoothing constant

## <span style='color: blue'>Calculating Parameters</span>

*Calculate the probabilities for each word in vocabulary for both spam and ham.*

*Store the results in dictionaries, one for spam, the other for ham.*

In [22]:
p_word_given_spam = {word: 0 for word in vocabulary}
p_word_given_ham = {word: 0 for word in vocabulary}

In [23]:
training_spam = training[training['Label'] == 'spam']
training_ham = training[training['Label'] == 'ham']

In [24]:
for word in vocabulary:
    n_word_given_spam = 0
    n_word_given_ham = 0
    for msg in training_spam['SMS']:
        n_word_given_spam += msg.count(word)
    for msg in training_ham['SMS']:
        n_word_given_ham += msg.count(word)
    p_word_given_spam[word] = (n_word_given_spam + alpha) / (n_spam + alpha * n_vocabulary)
    p_word_given_ham[word] = (n_word_given_ham + alpha) / (n_ham + alpha * n_vocabulary)

## <span style='color: blue'>Classifying a New Message</span>

*Create a function to classify a new message using the parameters previously calculated.*

In [25]:
def classify(msg, prnt=False):
    msg = (re.sub('\W', ' ', msg)
           .lower()
           .split())
    # Calculate probabilities that message is spam or ham
    p_spam_given_message = p_spam
    p_ham_given_message = p_ham
    for word in msg:
        if word in p_word_given_spam:
            p_spam_given_message *= p_word_given_spam[word]
        if word in p_word_given_ham:
            p_ham_given_message *= p_word_given_ham[word]
    if p_spam_given_message > p_ham_given_message:
        label = 'spam'
    elif p_spam_given_message < p_ham_given_message:
        label = 'ham'
    else:
        label = 'unclassified, please classify manually'
    if prnt:
        print('Label: {}.'.format(label))
    return label

*Test the function with an example of spam and ham.*

In [26]:
test_messages = [
    'WINNER!! This is the secret code to unlock the money: C3421.',
    'Sounds good, Tom, then see u there'
]

In [27]:
for test in test_messages:
    classify(test, True)

Label: spam.
Label: ham.


## <span style='color:blue'>Measuring the Spam Filter's Accuracy</span>

*Classify the messages in the testing dataset and calculate the accuracy metric.*

In [28]:
testing['Predicted'] = testing['SMS'].apply(classify)
testing['Correct'] = testing['Label'] == testing['Predicted']

In [29]:
n_tests = testing.shape[0]
n_correct = testing['Correct'].sum()
n_incorrect = n_tests - n_correct
accuracy = n_correct / n_tests

In [30]:
print('\nNumber of messages tested: {:,}'.format(n_tests))
print('Number of correct classifications: {:,}'.format(n_correct))
print('Number of incorrect classifications: {:,}'.format(n_incorrect))
print('Accuracy of model: {:.4f}\n'.format(accuracy))


Number of messages tested: 1,114
Number of correct classifications: 1,100
Number of incorrect classifications: 14
Accuracy of model: 0.9874



## <span style='color: blue'>Conclusion</span>

The accuracy of the model was 0.9874 which is very good meaning that there is a good chance that spam messages would be classified correctly by this algorithm.

Given the target for correct classifications was at least 80% correct identiifications, this algorith exceeds expectations.